In [0]:
%pip install kagglehub kagglesdk



Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import glob
import pandas as pd
import kagglehub

In [0]:
spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA staging")

DataFrame[]

In [0]:
path_download = kagglehub.dataset_download("victoremanuel/bigsus-sih-silver")

100%|██████████| 5.92G/5.92G [01:00<00:00, 105MB/s]

Extracting files...


# Busca arquivos com dados do RS

In [0]:
arquivos_rs = glob.glob(os.path.join(path_download, "**", "*RS*.parquet"), recursive=True)
if not arquivos_rs:
    arquivos_rs = glob.glob(os.path.join(path_download, "**", "*RS*.csv"), recursive=True)
if not arquivos_rs:
    arquivos_rs = glob.glob(os.path.join(path_download, "**", "*.parquet"), recursive=True)

caminho_arquivo = arquivos_rs[0]

In [0]:
if caminho_arquivo.endswith(".parquet"):
    df_pd = pd.read_parquet(caminho_arquivo)
else:
    df_pd = pd.read_csv(caminho_arquivo, low_memory=False, encoding='ISO-8859-1')

In [0]:
df_pd = df_pd.head(100000)

In [0]:
df_spark = spark.createDataFrame(df_pd)

In [0]:
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.staging.sih_raw_staging")